# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import pandas as pd


In [17]:
# Load raw starter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")


In [19]:
print(f"Dataset loaded successfully! Total rows: {len(df):,}")

# Selected candidate numeric features for clustering
clustering_features = [
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count'
]
print("\nSelected numeric feature types:")
print(df[clustering_features].dtypes)

Dataset loaded successfully! Total rows: 30,000

Selected numeric feature types:
impressions_90d             int64
clicks_90d                  int64
sessions_90d                int64
avg_position              float64
ctr                       float64
engagement_rate           float64
scroll_rate               float64
content_age_days            int64
days_since_last_update      int64
word_count                float64
dtype: object


In [23]:
# Audit feature matrix for data leakage columns
forbidden_cols = ['health_score', 'priority_score', 'is_declining_label', 'trend_direction', 'trend_pct']
leaked = [col for col in forbidden_cols if col in clustering_features]
assert len(leaked) == 0, f"Leakage violation! Found forbidden columns: {leaked}"

print("✅ Leakage audit passed: No forbidden columns in feature set.")

print("\nFeature summary statistics across key quantiles:")
print(df[clustering_features].describe(percentiles=[0.25, 0.5, 0.75]).round(2))

✅ Leakage audit passed: No forbidden columns in feature set.

Feature summary statistics across key quantiles:
       impressions_90d  clicks_90d  sessions_90d  avg_position       ctr  \
count         30000.00    30000.00      30000.00      30000.00  30000.00   
mean           5200.37       16.10         37.07         16.34      0.51   
std           16838.02       75.08        107.07         15.22      3.28   
min               1.00        0.00          1.00          0.00      0.00   
25%              81.00        0.00          2.00          6.20      0.00   
50%             731.00        1.00          7.00         10.80      0.07   
75%            3615.25        7.00         27.00         22.30      0.29   
max          517715.00     4178.00       4345.00        245.00    100.00   

       engagement_rate  scroll_rate  content_age_days  days_since_last_update  \
count         30000.00     29875.00          30000.00                30000.00   
mean              2.53        18.21       

In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prepare raw numeric features
X_raw = df[clustering_features].copy()
X_raw = X_raw.fillna(X_raw.median())

# Handle avg_position = 0 (unranked pages replaced with 100)
X_raw['avg_position'] = X_raw['avg_position'].replace(0, 100)

# Log-transform heavy-tailed count columns
skewed_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'word_count', 'days_since_last_update']
for col in skewed_cols:
    X_raw[col] = np.log1p(X_raw[col])

# Standardize features for distance calculations
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Evaluate baseline K-Means (K=5)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled[:5000], cluster_labels[:5000])
print(f"Baseline K-Means (K=5) Silhouette Score: {score:.4f}")
print(f"Cluster sizes distribution: {pd.Series(cluster_labels).value_counts().to_dict()}")

Baseline K-Means (K=5) Silhouette Score: 0.2059
Cluster sizes distribution: {4: 10049, 1: 8974, 2: 8866, 3: 1647, 0: 464}


In [27]:
# Verify table grain uniqueness
total_rows = len(df)
unique_ids = df['content_id'].nunique()

print(f"Total Rows in Dataset: {total_rows:,}")
print(f"Unique content_ids: {unique_ids:,}")
assert total_rows == unique_ids, "Grain error: Duplicate content_id entries found!"

print(f"\nProcessed Scaled Feature Matrix Shape: {X_scaled.shape}")
print("Sample scaled feature values (Row 1):", X_scaled[0].round(3))

Total Rows in Dataset: 30,000
Unique content_ids: 30,000

Processed Scaled Feature Matrix Shape: (30000, 10)
Sample scaled feature values (Row 1): [ 0.764  1.48   0.337 -0.443  0.076  0.403 -0.462 -0.521 -0.494  0.326]


In [29]:
# Demonstrate why a simple 1-variable rule fails
old_pages = df[df['content_age_days'] > 365]
champions = old_pages[(old_pages['avg_position'] > 0) & (old_pages['avg_position'] <= 10) & (old_pages['ctr'] > 1.0)]
decay_risk = old_pages[(old_pages['days_since_last_update'] > 180) & (old_pages['avg_position'] > 15)]

print(f"Total pages > 1 year old: {len(old_pages):,}")
print(f"  └─ Old pages that are Champions (Top rank + High CTR): {len(champions):,}")
print(f"  └─ Old pages at decay risk (Stale update + Poor rank): {len(decay_risk):,}")
print("\n=> Conclusion: A static rule ('age > 365 = stale') fails because old pages split into completely different archetypes!")

Total pages > 1 year old: 6,360
  └─ Old pages that are Champions (Top rank + High CTR): 115
  └─ Old pages at decay risk (Stale update + Poor rank): 2

=> Conclusion: A static rule ('age > 365 = stale') fails because old pages split into completely different archetypes!


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [22]:
"""My task type is **Unsupervised Content Clustering**. We use distance-based clustering (such as K-Means)
to discover natural performance archetypes across content items (`content_id`). The decision this improves is
enabling SEO editors to apply targeted bulk actions (protect champions, rewrite stale pages, optimize CTR for
hidden gems, or prune dead pages) across distinct clusters rather than inspecting 30,000 pages individually."""

'My task type is **Unsupervised Content Clustering**. We use distance-based clustering (such as K-Means) to discover natural performance archetypes across content items (`content_id`). The decision this improves is enabling SEO editors to apply targeted bulk actions (protect champions, rewrite stale pages, optimize CTR for hidden gems, or prune dead pages) across distinct clusters rather than inspecting 30,000 pages individually.'

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [20]:
"""Because this is an unsupervised clustering task, there is **no ground-truth target variable** used in model
 training. Instead, we formulate an **Archetype Hypothesis**
(5 expected performance profiles: Champions, Stale Visible Pages, Hidden Gems, Engagement Bottlenecks,
 and Low Demand Pages) to evaluate cluster quality after fitting. We strictly exclude future outcome labels (`is_declining_label`)
  and internal product scores (`health_score`, `priority_score`) from our feature set to prevent data leakage."""

'Because this is an unsupervised clustering task, there is **no ground-truth target variable** used in model\n training. Instead, we formulate an **Archetype Hypothesis** \n(5 expected performance profiles: Champions, Stale Visible Pages, Hidden Gems, Engagement Bottlenecks,\n and Low Demand Pages) to evaluate cluster quality after fitting. We strictly exclude future outcome labels (`is_declining_label`)\n  and internal product scores (`health_score`, `priority_score`) from our feature set to prevent data leakage.'

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [25]:
"""For unsupervised clustering, success is measured by **Silhouette Score** (aiming for $> 0.35$), **Inertia Elbow Analysis** to select an optimal $K$ (e.g., $K=4$ or $K=5$), and **Archetype Interpretability** (distinct, business-meaningful medians for impressions, position, engagement, and content age across clusters)."""

'For unsupervised clustering, success is measured by **Silhouette Score** (aiming for $> 0.35$), **Inertia Elbow Analysis** to select an optimal $K$ (e.g., $K=4$ or $K=5$), and **Archetype Interpretability** (distinct, business-meaningful medians for impressions, position, engagement, and content age across clusters).'

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [26]:
"""The decision grain for this task is **one row per unique content item (`content_id`)** over a 90-day observation window. Because distance-based clustering algorithms rely on Euclidean distance, we apply `np.log1p()` to heavy-tailed count columns (`impressions_90d`, `clicks_90d`, `sessions_90d`, `word_count`, `days_since_last_update`) and standardize all numerical features with `StandardScaler` to ensure balanced distance contributions across all features."""

'The decision grain for this task is **one row per unique content item (`content_id`)** over a 90-day observation window. Because distance-based clustering algorithms rely on Euclidean distance, we apply `np.log1p()` to heavy-tailed count columns (`impressions_90d`, `clicks_90d`, `sessions_90d`, `word_count`, `days_since_last_update`) and standardize all numerical features with `StandardScaler` to ensure balanced distance contributions across all features.'

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [28]:
"""A simple fixed rule (IF-statement) fails because content performance is multi-dimensional and continuous. High content age alone does not make a page stale — an old page with high impressions, top ranking, and high CTR is a Champion, whereas a younger page with high impressions but low engagement is an Engagement Bottleneck. Clustering simultaneously evaluates impressions, ranking, CTR, engagement, and age to discover natural archetypes far beyond what manual IF-ELSE rules can capture."""

'A simple fixed rule (IF-statement) fails because content performance is multi-dimensional and continuous. High content age alone does not make a page stale — an old page with high impressions, top ranking, and high CTR is a Champion, whereas a younger page with high impressions but low engagement is an Engagement Bottleneck. Clustering simultaneously evaluates impressions, ranking, CTR, engagement, and age to discover natural archetypes far beyond what manual IF-ELSE rules can capture.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.